The purpose of this notebook is to pull time series data imagery from earth engine. Additionally, logs are created with each image pulled from earth engine. Monitoring the logs as the imagery is being pulled is a helpful way to ensure images are still being pulled as the logs are updated in real time.

In [12]:
import os
from datetime import datetime

import ee
import geemap
import geopandas as gpd
import numpy as np
import pandas as pd

from glacierview import config, preprocess
from glacierview import rasters as read  # was helpers/read.py
from glacierview import viz as explore  # was helpers/explore.py
from glacierview.earthengine import EePull  # was ee_helpers


In [13]:
# Trigger the authentication flow
ee.Authenticate()
# Initialize the library
ee.Initialize()

In [14]:
data_label = "full_time_series_c02_t1_l2"
glacier_view_dir = str(config.REPO_ROOT)
glims_dir = os.path.join(glacier_view_dir, "src", "glims")
log_dir = os.path.join(glacier_view_dir,"src","earth_engine","data","ee_landing_zone",data_label, "logs")
image_dir = os.path.join(glacier_view_dir,"src","earth_engine","data","ee_landing_zone",data_label, "landsat")

In [15]:
ee_pull = EePull(log_dir, "geog_area_rollup_250")

In [16]:
df_sample = pd.read_csv(os.path.join(glims_dir, "data", "inference_samples", "geog_area_rollup_250.csv"))
glims_ids = df_sample.glims_id

In [18]:
glims_ids

0       G043844E42752N
1       G044477E42719N
2       G041493E43296N
3       G041523E43281N
4       G041626E43230N
             ...      
1019    G089265E33965N
1020    G079566E35871N
1021    G079097E36208N
1022    G087339E36494N
1023    G089775E35610N
Name: glims_id, Length: 1024, dtype: object

In [6]:
glims_bb_path = os.path.join(glims_dir,"data", "training_samples","glims_18k_bb.shp")
df = gpd.read_file(glims_bb_path)

In [7]:
df = df[df.glac_id.isin(glims_ids)]

In [8]:
#only use if you want to resume pulling the data
done_glims_ids = os.listdir(image_dir) + os.listdir('/Volumes/T7/GlacierView/ee_landing_zone/full_time_series_c02_t1_l2/landsat')
df = df[~df.glac_id.isin(done_glims_ids)]

# only use for finishing up what's missing
# imgs_per_glims_id = [(glims_id,len(os.listdir(os.path.join(image_dir, glims_id)))) for glims_id in os.listdir(os.path.join(image_dir)) if len(glims_id) > 9 ]
# remaining_glims_ids = [glims_id for glims_id,file_count in imgs_per_glims_id if file_count < 50]
# df = df[df.glac_id.isin(remaining_glims_ids)]

In [9]:
print(df.shape)

(7, 32)


In [10]:
start_date = datetime(1980,1,1)
end_date = datetime(2024,12,31)

In [11]:
for idx,row in enumerate(df.iterrows()):
    glims_id = row[1].glac_id
    bounding_box = eval(row[1].bboxes)
    out_dir = os.path.join(image_dir,glims_id)
    if not os.path.exists(out_dir):
        os.mkdir(out_dir)
    try:
        ee_pull.export_landsat_five_images(glims_id,
                                        bounding_box,
                                        start_date,
                                        end_date,
                                        out_dir)
    except Exception:
        pass
    try:
        ee_pull.export_landsat_seven_images(glims_id,
                                        bounding_box,
                                        start_date,
                                        end_date,
                                        out_dir)
    except Exception:
        pass
    try:
        ee_pull.export_landsat_eight_images(glims_id,
                                        bounding_box,
                                        start_date,
                                        end_date,
                                        out_dir)
    except Exception:
        pass

Number of images in this collection:  427
Generating URL ...
Please wait ...
Data downloaded to /Users/mattw/Desktop/projects/GlacierView/src/earth_engine/data/ee_landing_zone/full_time_series_c02_t1_l2/landsat/G290272E32505S/G290272E32505S_1984-10-14_L5_C02_T1_L2_SR.tif
Generating URL ...
Please wait ...
Data downloaded to /Users/mattw/Desktop/projects/GlacierView/src/earth_engine/data/ee_landing_zone/full_time_series_c02_t1_l2/landsat/G290272E32505S/G290272E32505S_1986-04-27_L5_C02_T1_L2_SR.tif
Generating URL ...
Please wait ...
Data downloaded to /Users/mattw/Desktop/projects/GlacierView/src/earth_engine/data/ee_landing_zone/full_time_series_c02_t1_l2/landsat/G290272E32505S/G290272E32505S_1986-09-18_L5_C02_T1_L2_SR.tif
Generating URL ...
Please wait ...
Data downloaded to /Users/mattw/Desktop/projects/GlacierView/src/earth_engine/data/ee_landing_zone/full_time_series_c02_t1_l2/landsat/G290272E32505S/G290272E32505S_1986-11-05_L5_C02_T1_L2_SR.tif
Generating URL ...
Please wait ...
Dat